In [1]:
import pandas as pd
import os

# =========================================================
# 1. Pfade definieren
# =========================================================
path_sc = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\Daten (Sensor_Community)\sensor_community_complete_may_august.csv"
path_kk_pm25 = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\Daten (Messstation WÜ)\PM2.5_Wuerzburg_2024.csv"
path_kk_pm10 = r"C:\Users\AD\Downloads\PM10_2024(3).xlsx"

# Neue kombinierte Ausgabedatei
path_out = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\new_Output\sensor_community_mit_kopfklinik_may_august.csv"

print("Lade Datensätze...")

# =========================================================
# 2. Sensor.Community Daten laden
# =========================================================
# Komma als Dezimaltrennzeichen beachten
df_sc = pd.read_csv(path_sc, sep=";", decimal=",")

# Zeitstempel einheitlich als Datetime parsen
df_sc['Zeitstempel'] = pd.to_datetime(df_sc['Zeitstempel'], errors='coerce')

# Koordinaten sicher als Float behandeln
for col in ['lon', 'lat']:
    df_sc[col] = df_sc[col].astype(str).str.replace(',', '.', regex=False)
    df_sc[col] = pd.to_numeric(df_sc[col], errors='coerce')

# =========================================================
# 3. Kopfklinik Daten laden & vorbereiten
# =========================================================
# PM2.5 CSV laden
df_kk_pm25 = pd.read_csv(path_kk_pm25, sep=";", decimal=".")
df_kk_pm25['Zeitpunkt'] = pd.to_datetime(df_kk_pm25['Zeitpunkt'], format='%d.%m.%Y %H:%M', errors='coerce')

# PM10 Excel laden
df_kk_pm10 = pd.read_excel(path_kk_pm10)
# Spalte in 'PM10' umbenennen
df_kk_pm10.rename(columns={'Würzburg/Kopfklinik': 'PM10'}, inplace=True)
df_kk_pm10['Zeitpunkt'] = pd.to_datetime(df_kk_pm10['Zeitpunkt'], errors='coerce')

# Beide Kopfklinik-Werte anhand des Zeitpunkts mergen
df_kk = pd.merge(df_kk_pm25, df_kk_pm10, on="Zeitpunkt", how="inner")

# =========================================================
# 4. Kopfklinik filtern und an SC-Struktur anpassen
# =========================================================
print("Filtere Kopfklinik auf Mai - August 2024 und formatiere Spalten...")

# Auf den exakten Zeitraum (01.05. bis 31.08.) zuschneiden
mask = (df_kk['Zeitpunkt'] >= '2024-05-01') & (df_kk['Zeitpunkt'] <= '2024-08-31 23:59:59')
df_kk = df_kk.loc[mask].copy()

# Spalten umbenennen, damit sie exakt zu SC passen
df_kk.rename(columns={
    'Zeitpunkt': 'Zeitstempel',
    'Lon': 'lon',
    'Lat': 'lat'
}, inplace=True)

# Offizielle ID "1" vergeben
df_kk['sensor_id'] = "1"

# Nur die relevanten Spalten in der exakt gleichen Reihenfolge behalten
df_kk = df_kk[['Zeitstempel', 'sensor_id', 'lon', 'lat', 'PM2.5', 'PM10']]

# =========================================================
# 5. Datensätze zusammenführen (Concat)
# =========================================================
print("Füge Sensor.Community und Kopfklinik zusammen...")
df_final = pd.concat([df_sc, df_kk], ignore_index=True)

# Um es sauber zu haben: Nach Sensor ID und Zeitstempel sortieren
df_final.sort_values(by=['sensor_id', 'Zeitstempel'], inplace=True)

# Zeitstempel wieder in das saubere String-Format 'YYYY-MM-DD HH:MM:SS' bringen
df_final['Zeitstempel'] = df_final['Zeitstempel'].dt.strftime('%Y-%m-%d %H:%M:%S')

# Speichern (mit Komma als Dezimalzeichen, wie in deiner originalen SC-Datei!)
df_final.to_csv(path_out, sep=";", decimal=",", index=False)

print("-" * 60)
print(f"ERFOLG! Kombinierter Datensatz gespeichert unter:\n{path_out}")
print("-" * 60)

Lade Datensätze...
Filtere Kopfklinik auf Mai - August 2024 und formatiere Spalten...
Füge Sensor.Community und Kopfklinik zusammen...
------------------------------------------------------------
ERFOLG! Kombinierter Datensatz gespeichert unter:
C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\new_Output\sensor_community_mit_kopfklinik_may_august.csv
------------------------------------------------------------


In [1]:
import pandas as pd
import os

# =========================================================
# 1. Pfade definieren
# =========================================================
path_sc = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\Daten (Sensor_Community)\sensor_community_complete_may_august.csv"
path_kk_pm25 = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\Daten (Messstation WÜ)\PM2.5_Wuerzburg_2024.csv"
path_kk_pm10 = r"C:\Users\AD\Downloads\PM10_2024(3).xlsx"

# Neue kombinierte Ausgabedatei
path_out = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\new_Output\sensor_community__kopfklinik_may_august.csv"

print("Lade Datensätze...")

# =========================================================
# 2. Sensor.Community Daten laden
# =========================================================
df_sc = pd.read_csv(path_sc, sep=";", decimal=",")

# Zeitstempel flexibel parsen
df_sc['Zeitstempel'] = pd.to_datetime(df_sc['Zeitstempel'], format='mixed', errors='coerce')

# Koordinaten sicher als Float behandeln
for col in ['lon', 'lat']:
    df_sc[col] = df_sc[col].astype(str).str.replace(',', '.', regex=False)
    df_sc[col] = pd.to_numeric(df_sc[col], errors='coerce')

# =========================================================
# 3. Kopfklinik Daten laden & vorbereiten
# =========================================================
# PM2.5 CSV laden (flexibles Datumsformat)
df_kk_pm25 = pd.read_csv(path_kk_pm25, sep=";", decimal=".")
df_kk_pm25['Zeitpunkt'] = pd.to_datetime(df_kk_pm25['Zeitpunkt'], dayfirst=True, format='mixed', errors='coerce')
df_kk_pm25['Zeitpunkt'] = df_kk_pm25['Zeitpunkt'].dt.floor('min')

# PM10 Excel laden
df_kk_pm10 = pd.read_excel(path_kk_pm10)
df_kk_pm10.rename(columns={'Würzburg/Kopfklinik': 'PM10'}, inplace=True)
df_kk_pm10['Zeitpunkt'] = pd.to_datetime(df_kk_pm10['Zeitpunkt'], dayfirst=True, format='mixed', errors='coerce')
df_kk_pm10['Zeitpunkt'] = df_kk_pm10['Zeitpunkt'].dt.floor('min')

# WICHTIG: how="left" nutzen, damit PM2.5-Daten NICHT gelöscht werden, falls PM10 fehlt
df_kk = pd.merge(df_kk_pm25, df_kk_pm10, on="Zeitpunkt", how="left")

# =========================================================
# 4. Kopfklinik filtern und an SC-Struktur anpassen
# =========================================================
print("Filtere Kopfklinik auf Mai - August 2024 und formatiere Spalten...")

mask = (df_kk['Zeitpunkt'] >= '2024-05-01') & (df_kk['Zeitpunkt'] <= '2024-08-31 23:59:59')
df_kk = df_kk.loc[mask].copy()

df_kk.rename(columns={
    'Zeitpunkt': 'Zeitstempel',
    'Lon': 'lon',
    'Lat': 'lat'
}, inplace=True)

df_kk['sensor_id'] = "1"

# Nur die relevanten Spalten in der exakten Reihenfolge behalten
df_kk = df_kk[['Zeitstempel', 'sensor_id', 'lon', 'lat', 'PM2.5', 'PM10']]

print(f"-> Kopfklinik Datensätze im Zeitraum (01.05. - 31.08.): {len(df_kk)} Zeilen")

# =========================================================
# 5. Datensätze zusammenführen (Concat)
# =========================================================
print("Füge Sensor.Community und Kopfklinik zusammen...")
df_final = pd.concat([df_sc, df_kk], ignore_index=True)

df_final.sort_values(by=['sensor_id', 'Zeitstempel'], inplace=True)

# Zeitstempel in einheitliches String-Format konvertieren
df_final['Zeitstempel'] = df_final['Zeitstempel'].dt.strftime('%Y-%m-%d %H:%M:%S')

# Speichern
df_final.to_csv(path_out, sep=";", decimal=",", index=False)

print("-" * 60)
print(f"ERFOLG! Kombinierter Datensatz gespeichert unter:\n{path_out}")
print("-" * 60)

Lade Datensätze...
Filtere Kopfklinik auf Mai - August 2024 und formatiere Spalten...
-> Kopfklinik Datensätze im Zeitraum (01.05. - 31.08.): 2952 Zeilen
Füge Sensor.Community und Kopfklinik zusammen...
------------------------------------------------------------
ERFOLG! Kombinierter Datensatz gespeichert unter:
C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\new_Output\sensor_community__kopfklinik_may_august.csv
------------------------------------------------------------
